<center><h1>Annotations for the Dataset Generator</h1></center>
<center><h4>Going over all the code from everyone and splitting it up into a legible and usable base to generate the artifical markets for the Black-Sholes Training</h4></center>

### Imports

In [2]:
import numpy as np
import pandas as pd
from scipy.stats import t as t_dist
from arch import arch_model
import yfinance as yf
import time
import os
import json
import hashlib
from cir_model import CIRModel
from scipy.stats import truncnorm, norm
import pywt

### Miscellaneous Functions and Global Variables for Reproducibility

In [4]:
MASTER_SEED = 42  # Change this to generate different datasets

def set_all_seeds(seed):
    """Set all random seeds for reproducibility"""
    np.random.seed(seed)
    print(f"All seeds set to: {seed}")

def compute_dataframe_hash(df, sample_size=10000):
    """
    Compute hash of dataframe to verify reproducibility.
    Uses a sample to avoid memory issues with large datasets.
    """
    if len(df) > sample_size:
        df_sample = df.sample(n=sample_size, random_state=42).sort_index()
    else:
        df_sample = df
    
    return hashlib.md5(pd.util.hash_pandas_object(df_sample).values).hexdigest()

def save_metadata(metadata, filename='dataset_metadata.json'):
    """Save generation metadata for verification"""
    with open(filename, 'w') as f:
        json.dump(metadata, f, indent=2)
    print(f"\nMetadata saved to {filename}")

#### Setting the seed in order to reproduce the same numbers

In [5]:
# Set seed before any random operations
set_all_seeds(MASTER_SEED)

start_date = "2019-01-01"
end_date = "2025-01-01"

All seeds set to: 42


### Downloading FTSE 100 Market Data and Fitting Garch model
Using a Garch(1,1)-t model, which is defined with returns as
$$r_t=\mu + \varepsilon_t$$
with the residuals
$$\varepsilon_t=\sigma_tz_t$$
and the resurcive variance equation
$$\sigma_t^2 = \omega + \alpha\varepsilon_{t-1}^2 + \beta \sigma_{t-1}^2$$
which put into infinite form is
$$\sigma_t^2=\frac{\omega}{1-\beta}+\sum_{k=1}^{\infty}\alpha\beta^{k-1}\varepsilon_{t-k}^2$$
Given $\sigma^2_t = $ conditional variance, and
$$z_t\sim \text{i.i.d. } t_v(0,1)$$
and the constraints
$$\omega > 0\\ \alpha \geq 0 \\ \beta \geq 0\\ \alpha + \beta < 1$$
So in total for parameters, we have
$$\theta = (\mu, \omega, \alpha, \beta, v)$$

In [8]:
data = yf.download("^FTSE", start=start_date, end=end_date)

if isinstance(data.columns, pd.MultiIndex):
    close_prices = data['Close']['^FTSE']
else:
    close_prices = data['Close']

S0 = float(close_prices.iloc[-1])

print(f"Downloaded {len(close_prices):,} trading days")
print(f"S_0 (initial stock price): £{S0:,.2f}")

# Fit GARCH(1,1)-t
returns_pct = 100 * np.log(close_prices / close_prices.shift(1)).dropna()

print("\nFitting GARCH(1,1)-t model...")
model = arch_model(returns_pct, vol='Garch', p=1, q=1, dist='t')
result = model.fit(disp='off')

mu = result.params['mu']
omega = result.params['omega']
alpha = result.params['alpha[1]']
beta = result.params['beta[1]']
nu = result.params['nu']

print(f"Parameters: μ={mu:.4f}%, ω={omega:.6f}, α={alpha:.4f}, β={beta:.4f}, ν={nu:.2f}")

[*********************100%***********************]  1 of 1 completed

Downloaded 1,514 trading days
S_0 (initial stock price): £8,173.00

Fitting GARCH(1,1)-t model...
Parameters: μ=0.0484%, ω=0.046704, α=0.1335, β=0.8205, ν=4.60


### Parameterizing a Cox–Ingersoll–Ross Model 
The CIR model is defined by the SDE
$$dr_t = \kappa(\theta - r_t)dt + \sigma\sqrt{r_t}dW_t$$
where $\kappa(\theta - r_t)$ serves as the drift term with a long-run mean of $\theta$ and a mean reversion speed of $\kappa$. $W_t$ is a standard Wiener process.

The volatility term $\sigma\sqrt{r_t}$ creates level dependent volatility, which dampens volatility when $r_t$ is close to 0. If the Feller condition is met, defined by
$$2\kappa \theta \geq \sigma ^2$$
then negative interest rates are impossible.

In the code we parameterize our CIR model on past daily SONIA data provided by the U.S. Federal Reserve, previously downloaded

In [9]:
# Getting a dataframe of historical interest rates
historical_IR_sonia = pd.read_csv("parameterize_data\IUDSOIA.csv")
historical_IR_sonia["observation_date"] = pd.to_datetime(historical_IR_sonia["observation_date"])
historical_ir_wanted = historical_IR_sonia[(historical_IR_sonia["observation_date"] >= start_date) & (historical_IR_sonia["observation_date"] <= end_date)].dropna()

print(historical_ir_wanted.head())
# Using the CIR Model Class in the repo
cir = CIRModel(historical_ir_wanted)
cir.calibrate(method='mle')

     observation_date  interest_rate
5739       2019-01-02         0.7044
5740       2019-01-03         0.7048
5741       2019-01-04         0.7046
5742       2019-01-07         0.7052
5743       2019-01-08         0.7052
Loaded 1515 data points
Date range: 2019-01-02 00:00:00 to 2024-12-31 00:00:00
Mean rate: 200.2999%
MLE optimization failed, falling back to method of moments

Calibrated parameters (Method of Moments):
  κ (kappa): 0.0557 - mean reversion speed
  θ (theta): 200.2999% - long-term mean
  σ (sigma): 0.4673 - volatility
  Feller condition (2κθ > σ²): 0.223076 > 0.218387 = True


{'kappa': 0.05568541895992372,
 'theta': 2.002999471947195,
 'sigma': 0.4673194433258763}

### Saving the model parameters in order to load them for later without using having to reparameterize the models

In [10]:
params = {
    "End Price" : S0,
    "Mean IR" : cir.historical_data['interest_rate'].mean(),
    "IR std" : np.std(cir.historical_data['interest_rate']),
    "n_days" : int(len(close_prices)),
    "GARCH" : {
        "mu": mu,
        "omega": omega,
        "alpha": alpha,
        "beta": beta,
        "nu": nu
    },
    "CIR": {
        "kappa": cir.kappa,
        "theta": cir.theta,
        "sigma": cir.sigma
    }
}

with open("parameterize_data/model_params.json", "w") as f:
    json.dump(params, f, indent=4)
    
del mu, omega, alpha, beta, nu, cir, data

print("model_params.json written successfully.")

model_params.json written successfully.


### Reloading the model simulation parameters, so code can be executed beyond this line

In [7]:
with open("parameterize_data/model_params.json", "r") as f:
    params = json.load(f)

# Historical Parameters
S0 = params["End Price"]
ir_mean = params["Mean IR"]
ir_std = params["IR std"]
n_days_downloaded = params["n_days"]


# GARCH parameters
mu    = params["GARCH"]["mu"]
omega = params["GARCH"]["omega"]
alpha = params["GARCH"]["alpha"]
beta  = params["GARCH"]["beta"]
nu     = params["GARCH"]["nu"]

# CIR parameters
kappa = params["CIR"]["kappa"]
theta = params["CIR"]["theta"]
sigma = params["CIR"]["sigma"]

print("Option Price")
print(f"  S0 = {S0}")

print("\nInterest Rates")
print(f"  Mean Interest Rate = {ir_mean}")
print(f"  Interest Rate Standard Deviation = {ir_std}")

print("\nGARCH Parameters:")
print(f"  mu    (μ) = {mu}")
print(f"  omega (ω) = {omega}")
print(f"  alpha (α) = {alpha}")
print(f"  beta  (β) = {beta}")
print(f"  nu    (ν) = {nu}")

print("\nCIR Parameters:")
print(f"  kappa (κ) = {kappa}")
print(f"  theta (θ) = {theta}")
print(f"  sigma (σ) = {sigma}")

Option Price
  S0 = 8173.0

Interest Rates
  Mean Interest Rate = 2.002999471947195
  Interest Rate Standard Deviation = 2.1061137312665377

GARCH Parameters:
  mu    (μ) = 0.04842674175412159
  omega (ω) = 0.04670376471231804
  alpha (α) = 0.1334992945018068
  beta  (β) = 0.8204776734410293
  nu    (ν) = 4.6037768961503795

CIR Parameters:
  kappa (κ) = 0.05568541895992372
  theta (θ) = 2.002999471947195
  sigma (σ) = 0.4673194433258763


### Setting up the Simulation Parameters and Saved Metadata for reproducibility 

In [8]:
n_simulations = 10000
T_maturity = 5  # All simulations start at 5 years
K_percentages = np.array([0.60, 0.70, 0.80, 0.90, 1.00, 1.10, 1.20, 1.30, 1.40, 1.50])
days_per_year = 252
n_days = (T_maturity - 1) * days_per_year + 1 # Total days for each simulation
mu_adjusted = 0.04
min_starting_ir = 0.01
burn_in_period = 90

print(f"Total simulations: {n_simulations:,}")
print(f"Maturity (T): {T_maturity} years for ALL simulations")
print(f"Days per simulation: {n_days}")
print(f"K choices: {K_percentages * 100}%")

# Store metadata for reproducibility
metadata = {
    'master_seed': MASTER_SEED,
    'n_simulations': int(n_simulations),
    'T_maturity': int(T_maturity),
    'K_percentages': K_percentages.tolist(),
    'S0': float(S0),
    'days_per_year': int(days_per_year),
    'n_days': int(n_days),
    'mu_adjusted': float(mu_adjusted),
    'burn_in_period' : burn_in_period,
    'data_download': {
        'ticker': '^FTSE',
        'start_date': start_date,
        'end_date': end_date,
        'n_days': n_days_downloaded
    },
    'GARCH_params': {
        "mu": mu,
        "omega": omega,
        "alpha": alpha,
        "beta": beta,
        "nu": nu
    },
    "CIR_params" : {
        "kappa": kappa,
        "theta": theta,
        "sigma": sigma
    },
    "IR_params" : {
        "IR Mean" : ir_mean,
        "IR Standard Deviation" : ir_std,
        "Minimum IR Start" : min_starting_ir
    },
    'python_version': os.sys.version,
    'numpy_version': np.__version__,
    'pandas_version': pd.__version__,
    'generation_timestamp': time.strftime('%Y-%m-%d %H:%M:%S')
}

Total simulations: 10,000
Maturity (T): 5 years for ALL simulations
Days per simulation: 1009
K choices: [ 60.  70.  80.  90. 100. 110. 120. 130. 140. 150.]%


### Creating Strike Prices for the Whole Dataset

In [9]:
# CRITICAL: Set seed before random assignments
np.random.seed(MASTER_SEED)

# Each simulation gets ONE randomly assigned strike price
K_pct_assignments = np.random.choice(K_percentages, size=n_simulations)
K_assignments = S0 * K_pct_assignments

print(f"K distribution:")
for k_pct in K_percentages:
    count = np.sum(K_pct_assignments == k_pct)
    print(f"  K={k_pct*100:.0f}%: {count:,} ({count/n_simulations*100:.1f}%)")

K distribution:
  K=60%: 1,053 (10.5%)
  K=70%: 985 (9.8%)
  K=80%: 996 (10.0%)
  K=90%: 971 (9.7%)
  K=100%: 962 (9.6%)
  K=110%: 1,021 (10.2%)
  K=120%: 1,017 (10.2%)
  K=130%: 967 (9.7%)
  K=140%: 994 (9.9%)
  K=150%: 1,034 (10.3%)


### Simulating GARCH Price paths for the whole dataset
What this code is doing is creating two seperate multidimensional arrays through numpy to track volatility and price over time. We have two matrices $S_{ij}$ and $\sigma_{ij}$. Where $i$ is the number of scenarios we are running and $j$ is the number of days per scenario. Our initial volatility is given by
$$ \sigma^2_0 = \frac{\omega}{1-\alpha - \beta} $$
which is the expectation of variance. We set our initial positions for the simulations at 
$$\sigma_{i0} = \sqrt{\sigma^2}, \text{ as defined above}$$
$$S_{i0}=S0, \text{ the end of the stock data price time series we parameterized the GARCH on}$$
A Monte Carlo simulation is now performed using the GARCH, where
$$z_{i,t} \sim t_v$$
Which we normalize so
$$\text{Var}(z_{i,t})=1$$
Which we do by letting
$$z_{i,t}=\frac{\tilde{z}_{i,t}}{\sqrt{\frac{v}{v-2}}}$$
Since the normal t distribution variance is given by
$$\text{Var}(t_v) = \frac{v}{v-2}$$
The rest is just calculating the current variables using the equations described earlier for GARCH. Two things to note however are that
1. Volatility is capped at 25%
2. An additive log return process is used for numerical stability. Where instead of a multiplicative process
$$S_t = S_{t-1}e^{r_t}$$
we use
$$\log S_t = \log S_{t-1}+r_t$$
where
$$r_t = \mu + \varepsilon_t$$
and at the very end we take
$$S_t = e^{\log S_t}$$
What this does is uses a additive process instead of a multiplicative process the whole time which avoids floating point errors that come heavy multiplication

In [10]:
def simulate_garch(S0, n_days, n_scenarios, mu, omega, alpha, beta, nu, seed=None, burn_in=90):
    """
    High-performance GARCH(1,1)-t simulation
    using log-prices for numerical stability,
    90-day burn-in, and final rounding to 4 decimal places.
    """
    
    if seed is not None:
        np.random.seed(seed)
    
    # Adding a burn in period
    total_days = n_days + burn_in + 1
    
    # Pre-draw standardized shocks for performance
    z = t_dist.rvs(df=nu, size=(n_scenarios, total_days))
    z /= np.sqrt(nu / (nu - 2)) # Standardizing 
    
    # Allocate arrays, multidimensional arrays
    var = np.empty((n_scenarios, total_days))
    log_S = np.empty((n_scenarios, total_days))
    
    # Initial conditions
    var0 = omega / (1 - alpha - beta)
    var[:, 0] = var0
    log_S[:, 0] = np.log(S0)
    
    
    # Time recursion
    for t in range(1, total_days):
        sigma_prev = np.sqrt(var[:, t-1])
        eps = sigma_prev * z[:, t]
        
        # Log-return update (additive)
        log_S[:, t] = log_S[:, t-1] + (mu + eps) / 100
        
        # Variance update
        var[:, t] = omega + alpha * eps**2 + beta * var[:, t-1]
        var[:, t] = np.minimum(var[:, t], 25.0) # Capping volatility 
    
    # Get final sigma
    sigma_final = np.sqrt(var[:,:])
    
    # Convert to prices
    S_final = np.exp(log_S)
    
    
    
    return S_final, sigma_final

### Simulating interest rates using CIR
Some notes about the methods:
1. The starting interest rate is sampled from a truncated normal distribution between (0.01, 25). 
$$X_{i,0} = \text{TruncNormal}(\mu_0, \sigma^2_0; 0.01, 25.0)$$
Which means it has a PDF of
$$f(x) = \frac{\phi\left(\frac{x-\mu_0}{\sigma_0}\right)}{\sigma\left[\Phi(b)-\Phi(a)\right]} \qquad \text{for $x \in [0.01, 10]$}$$
where in this case
$$a = \Phi\left(\frac{0.01-\mu_0}{\sigma_0}\right), \qquad b=\Phi\left(\frac{25-\mu_0}{\sigma_0}\right)$$
2. Using the Euler-Maruyama Method
$$dr_t = \kappa (\theta - r_t)dt+\sigma\sqrt{r_t}dW_t$$
we turn into
$$r_{t+1}=r_t+\kappa(\theta-r_t)\Delta t + \sigma\sqrt{r_t}\sqrt{\Delta t}Z_t$$
where
$$Z_t \sim N(0,1)$$

In [11]:
def simulate_CIR_paths(n_scenarios, n_steps, kappa, theta, sigma, mu0, sigma0_init, dt=1/252,burn_in_days=90, seed=None):
    if seed is not None:
        np.random.seed(seed)
    
    # Convert burn-in days to number of steps
    burn_in_steps = int(burn_in_days / (dt * 252)) if dt != 1/252 else burn_in_days
    
    print(f"Total burn in steps: {burn_in_steps}")
    
    total_steps = n_steps + burn_in_steps + 1
    
    # Getting the truncated normal samples
    lower, upper = 0.01, 25.0
    
    a = (lower - mu0) / sigma0_init
    b = (upper - mu0) / sigma0_init
    
    X0_samples = truncnorm.rvs(
        a, b,
        loc=mu0,
        scale=sigma0_init,
        size=n_scenarios
    )
    
    # Creating storage for the variables
    X_paths_full = np.zeros((n_scenarios, total_steps))
    X_paths_full[:, 0] = X0_samples
    X_current = X0_samples.copy()
    
    # Drawing all the nomral variables at once to speed up performance
    Z = np.random.normal(size=(n_scenarios, total_steps - 1))
    
    # Looping over the time steps
    for t in range(1, total_steps):
        # Getting z variable
        z = Z[:, t - 1]
        
        # Drift and diffusion variables
        drift = kappa * (theta - X_current) * dt
        diffusion = sigma * np.sqrt(np.maximum(X_current, 0.0)) * np.sqrt(dt) * z
        
        # Calculating next rate using current rate, drift, and diffusion
        X_next = X_current + drift + diffusion
        
        # Enforce positivity
        X_next = np.maximum(X_next, 0.0)
        
        # Updating current
        X_paths_full[:, t] = X_next
        X_current = X_next
    
    # Converting to decimal format
    X_paths_full = X_paths_full / 100.0
    
    return X_paths_full

### Running the Simulations

In [12]:
# =============================================================================
# Sorting out saving directory
# =============================================================================

output_dir = 'black_scholes_simulation_data'
os.makedirs(output_dir, exist_ok=True)

total_start = time.time()

print(f"\nGenerating {n_simulations:,} simulations × {n_days} days")

# =============================================================================
# Running GARCH
# =============================================================================

print(f"Running GARCH simulation...")
sim_start = time.time()

# CRITICAL: Use deterministic seed
simulation_seed = MASTER_SEED + 5000

S_paths, sigma_paths = simulate_garch(
    S0=S0,
    n_days=n_days,
    n_scenarios=n_simulations,
    mu=mu_adjusted,
    omega=omega,
    alpha=alpha,
    beta=beta,
    nu=nu,
    seed=simulation_seed,
    burn_in=burn_in_period
)

print(f"GARCH simulation completed in {time.time() - sim_start:.1f}s")

# =============================================================================
# Running CIR Model
# =============================================================================

print(f"Running CIR simulation...")
sim_start = time.time()

# Deterministic seed for CIR
cir_seed = MASTER_SEED + 7000

interest_paths = simulate_CIR_paths(
    n_scenarios=n_simulations,
    n_steps=n_days,
    kappa=kappa,
    theta=theta,
    sigma=sigma,
    mu0=ir_mean,
    sigma0_init=ir_std,
    dt=1/252,
    seed=cir_seed,
    burn_in_days=burn_in_period
)

print(f"CIR simulation completed in {time.time() - sim_start:.1f}s")

# =============================================================================
# Building a vectorized Dataset 
# =============================================================================

print(f"\nBuilding dataset...")
build_start = time.time()


n_rows = n_simulations * n_days 

# Create arrays for each column
simulation_col = np.repeat(np.arange(n_simulations), n_days + burn_in_period + 1)
day_col = np.tile(np.arange(-burn_in_period, n_days + 1), n_simulations)
S_col = S_paths.flatten()
K_col = np.repeat(K_assignments, n_days + burn_in_period + 1)
interest_col = interest_paths.flatten()

# Create T column:
# End at exactly 1 year remaining
T_sequence = (5 * days_per_year + burn_in_period - np.arange(n_days + burn_in_period + 1)) / days_per_year
T_col = np.tile(T_sequence, n_simulations)

# Creating a days till expiry 
T_sequence_expiry = np.arange(1350, 250, -1)
days_till_expiry = np.tile(T_sequence_expiry, n_simulations)

# Convert volatility to annualized percentage
sigma_col = sigma_paths.flatten() * np.sqrt(days_per_year) / 100

print(f"Arrays built in {time.time() - build_start:.1f}s")

# =============================================================================
# Creating the DataFrame
# =============================================================================

print(f"Creating DataFrame...")
df_start = time.time()

# Debugging
print(simulation_col.shape)
print(day_col.shape)
print(S_col.shape)
print(K_col.shape)
print(T_col.shape)
print(sigma_col.shape)
print(interest_col.shape)


df = pd.DataFrame({
    'simulation': simulation_col,
    'day': day_col,
    'days_till_expiry' : days_till_expiry,
    'S': S_col,
    'K': K_col,
    'T': T_col,
    'sigma': sigma_col,
    'r': interest_col
})

# Sort by simulation, then descending T (day ascending)
df = df.sort_values(
    by=['simulation', 'day'],
    ascending=[True, True]
).reset_index(drop=True)

print(f"DataFrame created in {time.time() - df_start:.1f}s")

total_time = time.time() - total_start
print(f"\n" + "=" * 70)
print(f"SIMULATION COMPLETE! Total time: {total_time / 60:.1f} minutes")
print("=" * 70)

# =============================================================================
# Saving
# =============================================================================

print("\n" + "=" * 70)
print("STEP 5: FINAL DATASET")
print("=" * 70)

print("\nStatistics:")
print(df.describe())

# Compute final hash
print("\nComputing dataset hash...")
final_hash = compute_dataframe_hash(df)
print(f"Final dataset hash: {final_hash}")

# Save final file as CSV
print("\nSaving final dataset...")
save_start = time.time()
df.to_csv('black_scholes_simulation_data_T5.csv', index=False)
save_time = time.time() - save_start

file_size_gb = os.path.getsize('black_scholes_simulation_data_T5.csv') / 1e9
print(f"Saved to black_scholes_simulation_data_T5.csv in {save_time:.1f}s")
print(f"File size: {file_size_gb:.2f} GB")

# =============================================================================
# STEP 7: SAVE METADATA AND VERIFICATION INFO
# =============================================================================

print("\n" + "=" * 70)
print("STEP 6: SAVING METADATA FOR REPRODUCIBILITY")
print("=" * 70)

# Add hashes and final statistics to metadata
metadata['final_hash'] = final_hash
metadata['total_rows'] = int(len(df))
metadata['file_size_gb'] = float(file_size_gb)
metadata['total_generation_time_minutes'] = float(total_time / 60)
metadata['simulation_seed'] = int(simulation_seed)
metadata['cir_seed'] = int(cir_seed)

# Save metadata
save_metadata(metadata, f'{output_dir}/dataset_metadata.json')

# Also save a verification file with just the hash
verification = {
    'master_seed': MASTER_SEED,
    'simulation_seed': simulation_seed,
    'final_hash': final_hash,
    'total_rows': int(len(df)),
    'n_simulations': int(n_simulations),
    'T_maturity': int(T_maturity),
    'n_days': int(n_days),
    'cir_seed' : int(cir_seed),
    'instructions': f'Run the generation script with MASTER_SEED={MASTER_SEED} to reproduce this exact dataset'
}

with open('REPRODUCIBILITY_INFO.json', 'w') as f:
    json.dump(verification, f, indent=2)

print(f"\nVerification info saved to REPRODUCIBILITY_INFO.json")

print("\n" + "=" * 70)
print("DONE!")
print("=" * 70)
print("\nTo reproduce this dataset on another computer:")
print(f"1. Use MASTER_SEED = {MASTER_SEED}")
print(f"2. Expected final hash: {final_hash}")
print(f"3. Expected total rows: {len(df):,}")
print(f"4. Expected file size: {file_size_gb:.2f} GB")
print("=" * 70)

# Clean up memory
del S_paths, sigma_paths, interest_paths


Generating 10,000 simulations × 1009 days
Running GARCH simulation...
GARCH simulation completed in 1.1s
Running CIR simulation...
Total burn in steps: 90
CIR simulation completed in 0.4s

Building dataset...
Arrays built in 0.1s
Creating DataFrame...
(11000000,)
(11000000,)
(11000000,)
(11000000,)
(11000000,)
(11000000,)
(11000000,)
DataFrame created in 0.7s

SIMULATION COMPLETE! Total time: 0.0 minutes

STEP 5: FINAL DATASET

Statistics:
         simulation           day  days_till_expiry             S  \
count  1.100000e+07  1.100000e+07      1.100000e+07  1.100000e+07   
mean   4.999500e+03  4.595000e+02      8.005000e+02  1.055791e+04   
std    2.886751e+03  3.175425e+02      3.175425e+02  3.088855e+03   
min    0.000000e+00 -9.000000e+01      2.510000e+02  2.386566e+03   
25%    2.499750e+03  1.847500e+02      5.257500e+02  8.446218e+03   
50%    4.999500e+03  4.595000e+02      8.005000e+02  9.769411e+03   
75%    7.499250e+03  7.342500e+02      1.075250e+03  1.189113e+04   
max

### Adding the Black-Scholes-Merton Price 
Price is given by:
$$C(S,t)=N(d_1)S-N(d_2)Ke^{-rT}$$
Where
1. $C(S,t)$ is the call price
2. $N()$ is the Normal CDF
3. $T$ is time to maturity
4. $S$ is stock price
5. $K$ is strike price
6. $r$ is the risk-free rate
7. $\sigma$ is volatility
and 
$$d_1 = \frac{\text{ln}\left(\frac{S}{K}\right)+\left(r+\frac{\sigma^2}{2}\right)T}{\sigma \sqrt{T}}$$
and
$$d_2 = d_1 - \sigma\sqrt{T}$$

In [13]:
def add_black_scholes_price(df):
    """
    Adds a Black-Scholes call price column ('bs_price') to the dataframe.
    
    Required columns:
    S, K, T, sigma, r
    
    r is assumed to be in percent and will be divided by 100.
    """
    
    r = df['r'] 
    S = df['S']
    K = df['K']
    T = df['T']
    sigma = df['sigma']
    
    # Avoid division by zero warnings
    epsilon = 1e-12
    sigma = np.maximum(sigma, epsilon)
    T = np.maximum(T, epsilon)
    
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    
    df['bs_price'] = (
        S * norm.cdf(d1) - 
        K * np.exp(-r * T) * norm.cdf(d2)
    )
    
    return df

input_file = "black_scholes_simulation_data_T5.csv"
output_file = "black_scholes_simulation_data_T5_with_price.csv"

chunksize = 1_000_000  # adjust based on memory

with pd.read_csv(input_file, chunksize=chunksize) as reader:
    for i, chunk in enumerate(reader):
        chunk = add_black_scholes_price(chunk)

        # Write header only for first chunk
        chunk.to_csv(
            output_file,
            mode='w' if i == 0 else 'a',
            header=(i == 0),
            index=False
        )

# Add-on Features Below
##### Features that modify the existing the variables to highlight features in the Black-Scholes variables

In [14]:
totaldf = pd.read_csv("black_scholes_simulation_data_T5_with_price.csv")
totaldf.head(115)

,simulation,day,days_till_expiry,S,K,T,sigma,r,bs_price
0,0,-90,1350,8173.000000,9807.6,5.357143,0.159915,0.031455,1152.325910
1,0,-89,1349,8194.737845,9807.6,5.353175,0.149432,0.032077,1096.775253
2,0,-88,1348,8127.574672,9807.6,5.349206,0.148336,0.032875,1065.130990
3,0,-87,1347,8181.645358,9807.6,5.345238,0.143305,0.032790,1055.308480
4,0,-86,1346,8126.365310,9807.6,5.341270,0.140573,0.032455,997.326131
...,...,...,...,...,...,...,...,...,...
110,0,20,1240,7891.812230,9807.6,4.920635,0.128184,0.025001,596.730212
111,0,21,1239,8042.057662,9807.6,4.916667,0.161621,0.025646,908.579728
112,0,22,1238,8073.581892,9807.6,4.912698,0.151737,0.025048,843.814893
113,0,23,1237,8111.747466,9807.6,4.908730,0.143855,0.024741,800.484906


#### Creates columns with the daily $\Delta$ for continuous variables S, $\sigma$, r, and black scholes price

In [15]:

def get_deltas(df: pd.DataFrame):
    # Making sure that the dataframe has the required column names to get deltas
    required = {"S", "sigma", "bs_price", "r", "simulation"}
    missing = required - set(df.columns)
    if missing:
        print(f"Missing required columns: {missing}")
        return

    # Initialise output arrays with zeros
    dS        = np.zeros(len(df))
    dsigma    = np.zeros(len(df))
    dBS_price = np.zeros(len(df))
    dr        = np.zeros(len(df))

    # Loop over each simulation by label rather than by integer position,
    # so that any removed rows (burn-in, day 1009) don't misalign the slices
    for sim_id, sim_df in df.groupby("simulation"):

        # Get the positional indices in df that correspond to this simulation's rows
        pos = df.index.get_indexer(sim_df.index)

        # .diff() is equivalent to x - x.shift(1); .values strips the pandas index
        # so the numpy assignment doesn't try to align on labels and silently zero-fill
        dS[pos]        = sim_df["S"].diff().values
        dsigma[pos]    = sim_df["sigma"].diff().values
        dBS_price[pos] = sim_df["bs_price"].diff().values
        dr[pos]        = sim_df["r"].diff().values

    # Assign the numpy arrays back to the dataframe
    df["dS"]        = dS
    df["dsigma"]    = dsigma
    df["dBS_price"] = dBS_price
    df["dr"]        = dr

    return df


get_deltas(totaldf)

,simulation,day,days_till_expiry,S,K,T,sigma,r,bs_price,dS,dsigma,dBS_price,dr
0,0,-90,1350,8173.000000,9807.6,5.357143,0.159915,0.031455,1152.325910,NaN,NaN,NaN,NaN
1,0,-89,1349,8194.737845,9807.6,5.353175,0.149432,0.032077,1096.775253,21.737845,-0.010482,-55.550657,0.000622
2,0,-88,1348,8127.574672,9807.6,5.349206,0.148336,0.032875,1065.130990,-67.163173,-0.001096,-31.644263,0.000798
3,0,-87,1347,8181.645358,9807.6,5.345238,0.143305,0.032790,1055.308480,54.070686,-0.005031,-9.822511,-0.000085
4,0,-86,1346,8126.365310,9807.6,5.341270,0.140573,0.032455,997.326131,-55.280048,-0.002733,-57.982349,-0.000334
...,...,...,...,...,...,...,...,...,...,...,...,...,...
10999995,9999,1005,255,14699.058794,8173.0,1.011905,0.105208,0.027421,6749.717550,-75.576557,0.001271,-80.450135,-0.000497
10999996,9999,1006,254,14713.723509,8173.0,1.007937,0.101344,0.028246,6770.129046,14.664715,-0.003864,20.411497,0.000825
10999997,9999,1007,253,14637.798161,8173.0,1.003968,0.103193,0.028897,6698.500861,-75.925347,0.001849,-71.628186,0.000651
10999998,9999,1008,252,14663.898670,8173.0,1.000000,0.099891,0.028500,6720.539768,26.100509,-0.003302,22.038907,-0.000397


#### Gets lagged and normal rolling means for continuous variables. The rolling means are 5, 15-5, and 30-10

In [16]:
def rolling_mean(x, window, lag=0):
    """
    Parameters
    ----------
    x : ndarray (1D)
    window : int
        Size of moving window
    lag : int
        Number of periods to lag the result
        lag=0 -> standard rolling mean
        lag=1 -> excludes current value
        lag=k -> shifted back by k periods

    Returns
    -------
    ndarray of same length as x
    """
    x = np.asarray(x, dtype=float)
    n = len(x)

    result = np.full(n, np.nan)

    if window <= 0:
        raise ValueError("window must be positive")
    if lag < 0:
        raise ValueError("lag must be >= 0")
    if window + lag > n:
        return result  # cannot compute anything

    # cumulative sum trick
    cumsum = np.cumsum(np.insert(x, 0, 0))

    # rolling sums (no lag yet)
    rolling_sum = cumsum[window:] - cumsum[:-window]
    rm = rolling_sum / window

    # place into correct positions with lag
    start = window - 1 + lag
    end = start + len(rm)

    result[start:end] = rm[:n - start]

    return result


def get_moving_averages(df: pd.DataFrame):
    """
    Computes rolling means per simulation over the full day range
    including the burn-in buffer. Negative days and day 1009 are left
    as-is and should be removed downstream.
    """
    required = {"S", "sigma", "bs_price", "r", "simulation", "day"}
    missing = required - set(df.columns)
    if missing:
        print(f"Missing required columns: {missing}")
        return

    configs = [(5, 0), (15, 5), (30, 10)]
    cols    = ['S', 'sigma', 'bs_price', 'r']
    col_map = {'bs_price': 'BSprice'}

    # Initialise output arrays with zeros
    ma = {(w, l, c): np.zeros(len(df)) for w, l in configs for c in cols}

    # Slice by simulation label so row-drops don't misalign boundaries
    for sim_id, sim_df in df.groupby("simulation"):
        idx = sim_df.index  # original index positions in df

        for w, l in configs:
            for c in cols:
                values = sim_df[c].values          # numpy array, includes burn-in
                rm     = rolling_mean(values, w, l) # NaN for uncomputable positions

                # Map results back to the correct rows in the output array
                # (using df.index.get_indexer so positional writes are safe)
                ma[(w, l, c)][df.index.get_indexer(idx)] = rm

    # Assign columns directly — no masking, negative days and day 1009 removed later
    for w, l in configs:
        for c in cols:
            col_name = f"ma_{col_map.get(c, c)}_window_{w}_lag_{l}"
            df[col_name] = ma[(w, l, c)]


get_moving_averages(totaldf)

In [17]:
totaldf.head(92)

,simulation,day,days_till_expiry,S,K,T,sigma,r,bs_price,dS,...,ma_BSprice_window_5_lag_0,ma_r_window_5_lag_0,ma_S_window_15_lag_5,ma_sigma_window_15_lag_5,ma_BSprice_window_15_lag_5,ma_r_window_15_lag_5,ma_S_window_30_lag_10,ma_sigma_window_30_lag_10,ma_BSprice_window_30_lag_10,ma_r_window_30_lag_10
0,0,-90,1350,8173.000000,9807.6,5.357143,0.159915,0.031455,1152.325910,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0,-89,1349,8194.737845,9807.6,5.353175,0.149432,0.032077,1096.775253,21.737845,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,0,-88,1348,8127.574672,9807.6,5.349206,0.148336,0.032875,1065.130990,-67.163173,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,0,-87,1347,8181.645358,9807.6,5.345238,0.143305,0.032790,1055.308480,54.070686,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,0,-86,1346,8126.365310,9807.6,5.341270,0.140573,0.032455,997.326131,-55.280048,...,1073.373353,0.032330,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
87,0,-3,1263,7718.199802,9807.6,5.011905,0.108024,0.026738,427.616413,-20.657234,...,480.741257,0.026678,7575.590234,0.136687,574.266923,0.027198,7666.070074,0.121323,549.681545,0.029586
88,0,-2,1262,7743.651759,9807.6,5.007937,0.105036,0.027106,421.629678,25.451957,...,466.474245,0.026770,7582.792757,0.136393,574.294197,0.027178,7651.588952,0.122440,547.716312,0.029396
89,0,-1,1261,7748.251611,9807.6,5.003968,0.101144,0.027365,400.464451,4.599852,...,442.285757,0.027061,7600.259109,0.135294,571.922369,0.027060,7639.238175,0.123104,543.930918,0.029222
90,0,0,1260,7788.934601,9807.6,5.000000,0.101772,0.027115,416.259465,40.682990,...,425.721818,0.027085,7619.553265,0.134313,571.082763,0.026949,7631.400505,0.123751,541.745867,0.029042


#### Gets the greek variables for options pricing

In [19]:
def add_greeks(df, option_type="call"):
    """
    Adds Greeks_Delta, Greeks_Gamma, Greeks_Vega,
    Greeks_Theta, Greeks_Rho columns to dataframe.
    
    Required columns:
        S, K, T, sigma, r
    """
    
    S = df["S"]
    K = df["K"]
    T = np.maximum(df["T"], 1e-12)   # avoid division by zero
    sigma = df["sigma"]
    r = df["r"]
    
    sqrtT = np.sqrt(T)
    
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * sqrtT)
    d2 = d1 - sigma * sqrtT
    
    # Delta
    df["Greeks_Delta"] = norm.cdf(d1)

    # Gamma
    df["Greeks_Gamma"] = norm.pdf(d1) / (S * sigma * sqrtT)

    # Vega (per 1.0 volatility change)
    df["Greeks_Vega"] = S * norm.pdf(d1) * sqrtT

    # Theta (per day)
    df["Greeks_Theta_daily"] = (
        - (S * norm.pdf(d1) * sigma) / (2 * sqrtT)
        - r * K * np.exp(-r * T) * norm.cdf(d2)
    ) / 365
    
    # Rho
    df["Greeks_Rho"] = K * T * np.exp(-r * T) * norm.cdf(d2)

add_greeks(totaldf)

In [20]:
totaldf.head(92)

,simulation,day,days_till_expiry,S,K,T,sigma,r,bs_price,dS,...,ma_r_window_15_lag_5,ma_S_window_30_lag_10,ma_sigma_window_30_lag_10,ma_BSprice_window_30_lag_10,ma_r_window_30_lag_10,Greeks_Delta,Greeks_Gamma,Greeks_Vega,Greeks_Theta_daily,Greeks_Rho
0,0,-90,1350,8173.000000,9807.6,5.357143,0.159915,0.031455,1152.325910,NaN,...,NaN,NaN,NaN,NaN,NaN,0.558731,0.000130,7464.790467,-0.599478,18290.277302
1,0,-89,1349,8194.737845,9807.6,5.353175,0.149432,0.032077,1096.775253,21.737845,...,NaN,NaN,NaN,NaN,NaN,0.559567,0.000139,7479.513748,-0.592609,18675.785444
2,0,-88,1348,8127.574672,9807.6,5.349206,0.148336,0.032875,1065.130990,-67.163173,...,NaN,NaN,NaN,NaN,NaN,0.554265,0.000142,7429.731271,-0.592041,18399.648965
3,0,-87,1347,8181.645358,9807.6,5.345238,0.143305,0.032790,1055.308480,54.070686,...,NaN,NaN,NaN,NaN,NaN,0.558655,0.000146,7464.591023,-0.589949,18790.714794
4,0,-86,1346,8126.365310,9807.6,5.341270,0.140573,0.032455,997.326131,-55.280048,...,NaN,NaN,NaN,NaN,NaN,0.546656,0.000150,7441.235871,-0.574598,18400.684159
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
87,0,-3,1263,7718.199802,9807.6,5.011905,0.108024,0.026738,427.616413,-20.657234,...,0.027198,7666.070074,0.121323,549.681545,0.029586,0.376151,0.000203,6558.404905,-0.374989,12407.440706
88,0,-2,1262,7743.651759,9807.6,5.007937,0.105036,0.027106,421.629678,25.451957,...,0.027178,7651.588952,0.122440,547.716312,0.029396,0.378208,0.000209,6588.580391,-0.375486,12555.313524
89,0,-1,1261,7748.251611,9807.6,5.003968,0.101144,0.027365,400.464451,4.599852,...,0.027060,7639.238175,0.123104,543.930918,0.029222,0.373214,0.000216,6562.461198,-0.368486,12466.341524
90,0,0,1260,7788.934601,9807.6,5.000000,0.101772,0.027115,416.259465,40.682990,...,0.026949,7631.400505,0.123751,541.745867,0.029042,0.380895,0.000215,6636.204946,-0.374509,12752.531985


#### Gets, bins, and stores the wavelet values for a previous 90 day window. Methodology is on a different markdown file

In [21]:
from joblib import Parallel, delayed
from numpy.lib.stride_tricks import sliding_window_view


# ── Constants ─────────────────────────────────────────────────────────────────
N_BINS = 7
WINDOW = 90
SCALES = np.logspace(np.log10(2), np.log10(45), N_BINS)
WAVELET_NAME = 'cmor1.5-1.0'
N_JOBS = 4


def process_simulation(sim_tuple):
    """
    For a single simulation, compute CWT-based energy features for every day
    that has a full WINDOW of history behind it.

    Returns the simulation dataframe with wavelet_bin_0 … wavelet_bin_48
    columns populated.

    Time bin convention:
      T0 = most recent steps  (narrow bin, small range)
      T6 = oldest steps       (wide bin,  large range)
    """
    sim, sim_df = sim_tuple

    sim_df = sim_df.sort_values("day").reset_index(drop=True).copy()

    # FIX 1: Isolate NaNs before CWT — NaNs propagate through convolution
    # and corrupt all windows that overlap with them.
    prices_raw      = sim_df["dBS_price"].values
    valid_mask      = ~np.isnan(prices_raw)
    valid_positions = np.where(valid_mask)[0]

    prices = prices_raw[valid_mask]
    n      = len(prices)

    feature_cols = [f"wavelet_bin_{i}" for i in range(N_BINS * N_BINS)]
    sim_df[feature_cols] = 0.0

    # FIX 2: Guard on valid row count, not raw row count
    if n < WINDOW:
        return sim_df

    # ── Step 1: CWT ───────────────────────────────────────────────────────────
    coeffs, freqs = pywt.cwt(prices, SCALES, WAVELET_NAME)
    energy = np.abs(coeffs) ** 2  # (n_scales, n_timepoints)

    # ── Step 2: Sliding windows ───────────────────────────────────────────────
    energy_windows = sliding_window_view(energy, WINDOW, axis=1)
    # shape: (n_scales, n_windows, WINDOW)
    # energy_windows[:, w, 0]        = oldest step in window w
    # energy_windows[:, w, WINDOW-1] = most recent step in window w

    # ── Step 3: Eligible indices ──────────────────────────────────────────────
    eligible_in_valid = np.arange(WINDOW - 1, n)
    window_positions  = eligible_in_valid - (WINDOW - 1)
    energy_windows    = energy_windows[:, window_positions, :]

    # FIX 3: Remap back to sim_df positional indices
    eligible_idx = valid_positions[eligible_in_valid]

    # ── Step 4: Bin edges ─────────────────────────────────────────────────────
    periods = 1.0 / freqs
    p_edges = np.logspace(np.log10(periods.min()), np.log10(periods.max()), N_BINS + 1)

    # Time axis inside a window: index 0 = oldest, index WINDOW-1 = most recent.
    # We want:
    #   recent steps (index near WINDOW-1) → narrow bins → T0
    #   old steps    (index near 0)        → wide bins   → T6
    #
    # Define recency = WINDOW - t  for t in [1..WINDOW]:
    #   t=WINDOW (most recent) → recency = 0  ... shifted to 1 for log
    #   t=1      (oldest)      → recency = WINDOW-1
    #
    # Use recency+1 so the log is defined, build log edges over [1, WINDOW],
    # then digitize: low recency+1 (recent) → bin 0 (narrow), high → bin 6 (wide).
    times    = np.arange(1, WINDOW + 1)           # 1=oldest slice, WINDOW=newest slice
    recency  = (WINDOW + 1) - times               # WINDOW=oldest, 1=newest
    r_edges  = np.logspace(np.log10(1), np.log10(WINDOW), N_BINS + 1)

    # bin 0 = recency in [1, r1]  → most recent steps (narrow)
    # bin 6 = recency in [r6, WINDOW] → oldest steps (wide)
    t_bin_idx = np.clip(np.digitize(recency, r_edges) - 1, 0, N_BINS - 1)

    p_bin_idx = np.clip(np.digitize(periods, p_edges) - 1, 0, N_BINS - 1)

    # ── Step 5: Aggregate into N_BINS × N_BINS feature matrix ─────────────────
    n_windows = energy_windows.shape[1]
    features  = np.zeros((n_windows, N_BINS * N_BINS))

    for i in range(N_BINS):
        scale_mask = (p_bin_idx == i)
        if not np.any(scale_mask):
            continue
        scale_slice = energy_windows[scale_mask]  # (n_scales_in_bin, n_windows, WINDOW)

        for j in range(N_BINS):
            time_mask = (t_bin_idx == j)
            if not np.any(time_mask):
                continue
            bin_energy = scale_slice[:, :, time_mask].mean(axis=(0, 2))  # (n_windows,)
            features[:, i * N_BINS + j] = bin_energy

    # ── Step 6: Write features back ───────────────────────────────────────────
    col_positions = sim_df.columns.get_indexer(feature_cols)
    sim_df.iloc[eligible_idx, col_positions] = features

    return sim_df


def compute_wavelet_parallel(df):
    """
    Split the dataframe by simulation, process each in parallel, then
    reassemble in original index order.
    """
    grouped = list(df.groupby("simulation"))

    results = Parallel(n_jobs=N_JOBS, backend="loky")(
        delayed(process_simulation)(group)
        for group in grouped
    )

    return pd.concat(results).sort_index()


totaldf = compute_wavelet_parallel(totaldf)

In [22]:
totaldf = totaldf.sort_values(by=["simulation", "day"], ascending=[True, True])
totaldf.head(92)

,simulation,day,days_till_expiry,S,K,T,sigma,r,bs_price,dS,...,wavelet_bin_39,wavelet_bin_40,wavelet_bin_41,wavelet_bin_42,wavelet_bin_43,wavelet_bin_44,wavelet_bin_45,wavelet_bin_46,wavelet_bin_47,wavelet_bin_48
0,0,-90,1350,8173.000000,9807.6,5.357143,0.159915,0.031455,1152.325910,NaN,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,0,-89,1349,8194.737845,9807.6,5.353175,0.149432,0.032077,1096.775253,21.737845,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,0,-88,1348,8127.574672,9807.6,5.349206,0.148336,0.032875,1065.130990,-67.163173,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,0,-87,1347,8181.645358,9807.6,5.345238,0.143305,0.032790,1055.308480,54.070686,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,0,-86,1346,8126.365310,9807.6,5.341270,0.140573,0.032455,997.326131,-55.280048,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
87,0,-3,1263,7718.199802,9807.6,5.011905,0.108024,0.026738,427.616413,-20.657234,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
88,0,-2,1262,7743.651759,9807.6,5.007937,0.105036,0.027106,421.629678,25.451957,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
89,0,-1,1261,7748.251611,9807.6,5.003968,0.101144,0.027365,400.464451,4.599852,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
90,0,0,1260,7788.934601,9807.6,5.000000,0.101772,0.027115,416.259465,40.682990,...,1436.478793,570.612510,415.725460,10.606126,422.634715,362.314093,422.039098,487.808413,674.419753,889.067310


##### Normalizing using min-max scaling across all wavelet columns at once since they are in the same space

In [23]:
wavelet_cols = [f'wavelet_bin_{i}' for i in range(49)]
totaldf[wavelet_cols] = (totaldf[wavelet_cols] - totaldf[wavelet_cols].min()) / (totaldf[wavelet_cols].max() - totaldf[wavelet_cols].min())

In [24]:
totaldf.head(92)

,simulation,day,days_till_expiry,S,K,T,sigma,r,bs_price,dS,...,wavelet_bin_39,wavelet_bin_40,wavelet_bin_41,wavelet_bin_42,wavelet_bin_43,wavelet_bin_44,wavelet_bin_45,wavelet_bin_46,wavelet_bin_47,wavelet_bin_48
0,0,-90,1350,8173.000000,9807.6,5.357143,0.159915,0.031455,1152.325910,NaN,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,0,-89,1349,8194.737845,9807.6,5.353175,0.149432,0.032077,1096.775253,21.737845,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,0,-88,1348,8127.574672,9807.6,5.349206,0.148336,0.032875,1065.130990,-67.163173,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,0,-87,1347,8181.645358,9807.6,5.345238,0.143305,0.032790,1055.308480,54.070686,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,0,-86,1346,8126.365310,9807.6,5.341270,0.140573,0.032455,997.326131,-55.280048,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
87,0,-3,1263,7718.199802,9807.6,5.011905,0.108024,0.026738,427.616413,-20.657234,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
88,0,-2,1262,7743.651759,9807.6,5.007937,0.105036,0.027106,421.629678,25.451957,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
89,0,-1,1261,7748.251611,9807.6,5.003968,0.101144,0.027365,400.464451,4.599852,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
90,0,0,1260,7788.934601,9807.6,5.000000,0.101772,0.027115,416.259465,40.682990,...,0.001222,0.000503,0.000398,0.000009,0.000461,0.000407,0.000529,0.000605,0.000876,0.001244


#### Creating a target variable that is just the bs_price shifted one day previous

In [25]:
totaldf['target_price'] = totaldf['bs_price'].shift(-1)

In [26]:
totaldf.head()

,simulation,day,days_till_expiry,S,K,T,sigma,r,bs_price,dS,...,wavelet_bin_40,wavelet_bin_41,wavelet_bin_42,wavelet_bin_43,wavelet_bin_44,wavelet_bin_45,wavelet_bin_46,wavelet_bin_47,wavelet_bin_48,target_price
0,0,-90,1350,8173.000000,9807.6,5.357143,0.159915,0.031455,1152.325910,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1096.775253
1,0,-89,1349,8194.737845,9807.6,5.353175,0.149432,0.032077,1096.775253,21.737845,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1065.130990
2,0,-88,1348,8127.574672,9807.6,5.349206,0.148336,0.032875,1065.130990,-67.163173,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1055.308480
3,0,-87,1347,8181.645358,9807.6,5.345238,0.143305,0.032790,1055.308480,54.070686,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,997.326131
4,0,-86,1346,8126.365310,9807.6,5.341270,0.140573,0.032455,997.326131,-55.280048,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1132.634101


#### Trimming off the days that are used for moving averages, waves, and target variables

In [27]:
totaldf = totaldf[(totaldf['day'] >= 0) & (totaldf['day'] < 1009)]
totaldf = totaldf.reset_index(drop=True)

In [28]:
pd.set_option('display.max_rows', 500)

In [29]:
print(totaldf.shape)
totaldf.head(1011)

(10090000, 80)


,simulation,day,days_till_expiry,S,K,T,sigma,r,bs_price,dS,...,wavelet_bin_40,wavelet_bin_41,wavelet_bin_42,wavelet_bin_43,wavelet_bin_44,wavelet_bin_45,wavelet_bin_46,wavelet_bin_47,wavelet_bin_48,target_price
0,0,0,1260,7788.934601,9807.6,5.000000,0.101772,0.027115,416.259465,40.682990,...,0.000503,0.000398,0.000009,0.000461,0.000407,0.000529,0.000605,0.000876,0.001244,395.518217
1,0,1,1259,7758.566777,9807.6,4.996032,0.101484,0.026563,395.518217,-30.367824,...,0.000530,0.000396,0.000960,0.000395,0.000401,0.000514,0.000611,0.000786,0.001261,371.289060
2,0,2,1258,7739.706462,9807.6,4.992063,0.099485,0.026244,371.289060,-18.860315,...,0.000575,0.000393,0.000140,0.000653,0.000531,0.000481,0.000599,0.000805,0.001260,344.081885
3,0,3,1257,7731.784705,9807.6,4.988095,0.096776,0.025686,344.081885,-7.921757,...,0.000614,0.000393,0.000517,0.000742,0.000320,0.000548,0.000595,0.000755,0.001270,414.702014
4,0,4,1256,7795.315086,9807.6,4.984127,0.104399,0.025592,414.702014,63.530381,...,0.000646,0.000388,0.000472,0.000443,0.000715,0.000520,0.000544,0.000760,0.001269,383.835069
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1006,0,1006,254,18069.359785,9807.6,1.007937,0.193002,0.019355,8451.622268,231.646607,...,0.002608,0.000744,0.010057,0.012447,0.017466,0.017283,0.019903,0.026857,0.040735,8251.336706
1007,0,1007,253,17876.002345,9807.6,1.003968,0.189548,0.018716,8251.336706,-193.357440,...,0.002628,0.000765,0.006788,0.015815,0.011928,0.018069,0.020707,0.025530,0.040751,8406.529869
1008,0,1008,252,18032.215183,9807.6,1.000000,0.181586,0.018705,8406.529869,156.212838,...,0.002540,0.000845,0.014102,0.011357,0.015765,0.017359,0.018866,0.026217,0.040323,8666.681312
1009,1,0,1260,8507.966398,7355.7,5.000000,0.106960,0.034308,2386.240185,-58.720777,...,0.000294,0.000142,0.000670,0.000912,0.000967,0.001239,0.001317,0.001366,0.001103,2367.867351


### Saving

In [33]:
totaldf.shape

(10090000, 80)

In [34]:
totaldf.to_csv("black_scholes_simulation_data_T5_full.csv", index=False)

In [2]:
import pandas as pd
totaldf = pd.read_csv("black_scholes_simulation_data_T5_full.csv")
totaldf.head(100)

,simulation,day,days_till_expiry,S,K,T,sigma,r,bs_price,dS,...,wavelet_bin_40,wavelet_bin_41,wavelet_bin_42,wavelet_bin_43,wavelet_bin_44,wavelet_bin_45,wavelet_bin_46,wavelet_bin_47,wavelet_bin_48,target_price
0,0,0,1260,7788.934601,9807.6,5.000000,0.101772,0.027115,416.259465,40.682990,...,0.000503,0.000057,0.000009,0.000461,0.000407,0.000536,0.000617,0.000469,0.000133,395.518217
1,0,1,1259,7758.566777,9807.6,4.996032,0.101484,0.026563,395.518217,-30.367824,...,0.000530,0.000057,0.000960,0.000395,0.000401,0.000522,0.000624,0.000421,0.000135,371.289060
2,0,2,1258,7739.706462,9807.6,4.992063,0.099485,0.026244,371.289060,-18.860315,...,0.000575,0.000056,0.000140,0.000653,0.000531,0.000488,0.000611,0.000431,0.000135,344.081885
3,0,3,1257,7731.784705,9807.6,4.988095,0.096776,0.025686,344.081885,-7.921757,...,0.000614,0.000056,0.000517,0.000742,0.000320,0.000556,0.000607,0.000404,0.000136,414.702014
4,0,4,1256,7795.315086,9807.6,4.984127,0.104399,0.025592,414.702014,63.530381,...,0.000646,0.000056,0.000472,0.000443,0.000715,0.000528,0.000554,0.000407,0.000136,383.835069
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,0,95,1165,8059.753921,9807.6,4.623016,0.155235,0.025102,807.909945,-20.735715,...,0.001253,0.000105,0.001016,0.001311,0.001432,0.001335,0.001575,0.000820,0.000121,756.964154
96,0,96,1164,8081.493974,9807.6,4.619048,0.145347,0.025667,756.964154,21.740053,...,0.001214,0.000108,0.000670,0.000894,0.001891,0.001320,0.001337,0.000893,0.000118,691.808881
97,0,97,1163,8086.744258,9807.6,4.615079,0.136060,0.025465,691.808881,5.250284,...,0.001165,0.000112,0.000724,0.001136,0.001172,0.001366,0.001560,0.000864,0.000121,643.536347
98,0,98,1162,8108.096707,9807.6,4.611111,0.128585,0.025048,643.536347,21.352449,...,0.001090,0.000119,0.001362,0.000940,0.001369,0.001398,0.001325,0.000903,0.000124,612.041827


In [3]:
totaldf.columns

Index(['simulation', 'day', 'days_till_expiry', 'S', 'K', 'T', 'sigma', 'r',
       'bs_price', 'dS', 'dsigma', 'dBS_price', 'dr', 'ma_S_window_5_lag_0',
       'ma_sigma_window_5_lag_0', 'ma_BSprice_window_5_lag_0',
       'ma_r_window_5_lag_0', 'ma_S_window_15_lag_5',
       'ma_sigma_window_15_lag_5', 'ma_BSprice_window_15_lag_5',
       'ma_r_window_15_lag_5', 'ma_S_window_30_lag_10',
       'ma_sigma_window_30_lag_10', 'ma_BSprice_window_30_lag_10',
       'ma_r_window_30_lag_10', 'Greeks_Delta', 'Greeks_Gamma', 'Greeks_Vega',
       'Greeks_Theta_daily', 'Greeks_Rho', 'wavelet_bin_0', 'wavelet_bin_1',
       'wavelet_bin_2', 'wavelet_bin_3', 'wavelet_bin_4', 'wavelet_bin_5',
       'wavelet_bin_6', 'wavelet_bin_7', 'wavelet_bin_8', 'wavelet_bin_9',
       'wavelet_bin_10', 'wavelet_bin_11', 'wavelet_bin_12', 'wavelet_bin_13',
       'wavelet_bin_14', 'wavelet_bin_15', 'wavelet_bin_16', 'wavelet_bin_17',
       'wavelet_bin_18', 'wavelet_bin_19', 'wavelet_bin_20', 'wavelet_bin_21'

In [4]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots


def plot_simulation(df: pd.DataFrame, simulation_number):
    """
    Plot bs_price, S, sigma, r for a given simulation
    on four stacked Plotly subplots.

    Parameters
    ----------
    df : pd.DataFrame
        Must contain columns:
        ['simulation', 'day', 'bs_price', 'S', 'sigma', 'r']
    simulation_number : int or str
        Simulation identifier to plot
    """

    # --- filter simulation ---
    sim_df = df[df["simulation"] == simulation_number].copy()

    if sim_df.empty:
        raise ValueError(f"No data found for simulation {simulation_number}")

    # sort by day to ensure proper line plotting
    sim_df = sim_df.sort_values("day")

    # --- create stacked subplots ---
    fig = make_subplots(
        rows=4,
        cols=1,
        shared_xaxes=True,
        vertical_spacing=0.04,
        subplot_titles=("bs_price", "S", "sigma", "r"),
    )

    columns = ["bs_price", "S", "sigma", "r"]

    # --- add traces ---
    for i, col in enumerate(columns, start=1):
        fig.add_trace(
            go.Scatter(
                x=sim_df["day"],
                y=sim_df[col],
                mode="lines",
                name=col,
            ),
            row=i,
            col=1,
        )

    # --- layout ---
    fig.update_layout(
        height=900,
        title=f"Simulation {simulation_number}",
        showlegend=False,
        hovermode="x unified",
    )

    fig.update_xaxes(title_text="day", row=4, col=1)

    return fig

In [14]:
fig = plot_simulation(totaldf, simulation_number=30)
fig.show()